In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader



In [ ]:
# Load dataset
df = pd.read_csv("/content/breast_cancer.csv")
df.head()


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


In [ ]:
# Features and target
X = df.drop('target', axis=1)
y = df['target']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
y_train.shape, y_test.shape

((455,), (114,))

In [ ]:


# Convert to tensors
X_train_tensor = torch.FloatTensor(X_train)
y_train_tensor = torch.FloatTensor(y_train.values).view(-1, 1)
X_test_tensor = torch.FloatTensor(X_test)
y_test_tensor = torch.FloatTensor(y_test.values).view(-1, 1)

# DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)


In [ ]:
X_train_tensor.shape, y_train_tensor.shape

(torch.Size([455, 30]), torch.Size([455, 1]))

## Define the Neural Network

In [ ]:
class BreastCancerANN(nn.Module):

    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(X_train.shape[1], 64)
        self.relu1 = nn.ReLU()

        self.fc2 = nn.Linear(64, 32)
        self.relu2 = nn.ReLU()

        self.fc3 = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):

        # First Hidden Layer
        sum_L1 = self.fc1(x)
        output_L1 = self.relu1(sum_L1)

        # Second Hidden Layer
        sum_L2 = self.fc2(output_L1)
        output_L2 = self.relu2(sum_L2)

        # Output Layer
        sum_L3 = self.fc3(output_L2)
        final_output = self.sigmoid(sum_L3)

        return final_output


model = BreastCancerANN()

print(model)

BreastCancerANN(
  (fc1): Linear(in_features=30, out_features=64, bias=True)
  (relu1): ReLU()
  (fc2): Linear(in_features=64, out_features=32, bias=True)
  (relu2): ReLU()
  (fc3): Linear(in_features=32, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


## Train the Model

In [ ]:
loss_fn= nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 50
for epoch in range(epochs):
    for xb, yb in train_loader:
        pred = model(xb)
        loss = loss_fn(pred, yb)
        optimizer.zero_grad() # 1. Clear old gradients
        loss.backward() # 2. Calculate gradients
        optimizer.step() # 3. Update weights

    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")


Epoch 10/50, Loss: 0.0056
Epoch 20/50, Loss: 0.0003
Epoch 30/50, Loss: 0.0011
Epoch 40/50, Loss: 0.0027
Epoch 50/50, Loss: 0.0000


## Evaluate the Model

In [ ]:
import torch
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# 1. Evaluation Mode
model.eval()

# 2. Disable Gradient Calculation
with torch.no_grad():

    # 3. Predictions
    outputs = model(X_test_tensor)

    # 4. Get Predicted Classes
    # y_pred = torch.argmax(outputs, dim=1)
    y_pred = outputs.round()

    # 5. Convert to NumPy
    y_pred = y_pred.numpy()
    y_true = y_test_tensor.numpy()

# 6. Evaluation Metrics
print("Accuracy:", accuracy_score(y_true, y_pred))


Accuracy: 0.9736842105263158
